In [3]:

import os, io, time, json, hashlib, pathlib, sys
import requests
import pandas as pd
import numpy as np
from dotenv import load_dotenv
import re
import matplotlib.pyplot as plt
from urllib.parse import urlparse
from datetime import datetime, timedelta
from config import *
from functions import *

def build_returns_matrix_in_chf(
    holdings: list[dict],
    lookback_days: int = 725,
    max_age: int = 24,
    no_fx: bool = False,
    usd_shift: bool = False,
    DEBUG: bool = False,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series]:
    """
    Build CHF daily returns matrix for the provided holdings.

        holdings: list of dicts with tickers:
      - name: row/column label in outputs
      - symbol: EODHD symbol (e.g., 'SWDA.LON', 'IBM')
      - ccy: base currency of the asset price series (GBP/USD/EUR/CHF)
      - gbx: bool; if True, divide close by 100.0 (LSE pence)
      - position: number of shares held (float)

        Returns:
      rets_df: DataFrame of CHF log returns, T x N
      prices_df: DataFrame of CHF closes, T x N
      weights: Series aligned to columns in rets_df
    """

    # Pre-fetch FX once per currency (excluding CHF)
    fx_map: dict[str, pd.Series] = {}
    needed_ccy = sorted({h["ccy"].upper() for h in holdings if h["ccy"].upper() != "CHF"})
    print('-------------fetching currencies-------------')
    for ccy in needed_ccy:
        ticker = f'{ccy}CHF.FOREX'
        params = {
            'from': START, 
            'to': today,
            'api_token': EOD_API    
        }
        url_fx = f'https://eodhd.com/api/eod/{ticker}'
        
        # Fetch EODHD daily FX and build a Series
        fx_df = fetch_csv_robust(url_fx, params=params, ticker=ticker, max_age=max_age)
        # Normalize and pick close
        fx_s = sort_cols(fx_df).rename(f"{ccy}CHF")

        if (ccy == "USD" and not no_fx and usd_shift):
            fx_s = shift_usd_fx_next_day(fx_s)
            print("    Applied USDCHF T+1 shift")

        fx_map[f'{ccy}CHF'] = fx_s

    if DEBUG:
        print(f">>>FX keys loaded: {list(fx_map.keys())}")

    # Build CHF close series per asset
    chf_close = {}
    series_ranges: dict[str, tuple[pd.Timestamp, pd.Timestamp, int]] = {}
    print('-------------fetching assets-------------')
    for h in holdings:
        if h.get("type", "").lower() == "cash":
            continue
        name  = h["name"]
        sym   = h["symbol"]
        ccy   = h["ccy"].upper()
        gbx   = h["gbx"]

        url_px = f'https://eodhd.com/api/eod/{sym}'

        # origage = max_age
        # if name == 'VEU':
        #     print('VEU:::')
        #     max_age = 0.001

        px_df = fetch_csv_robust(url_px, params=params, ticker=sym, max_age=max_age)

        # max_age = origage

        # Normalize, de-dup, and pick close-like series
        close_local = sort_cols(px_df)
        if gbx:
            close_local = close_local / 100.0
        if 'VEU.US' in name:
            close_local = close_local * 0.9
            print('VEU (XWMX proxy) * 0.9 to remove EM element')

        if (ccy == "CHF") or no_fx or h.get('risk_fx', '').lower() == 'none':
            print("    Skipping FX conversion for", name)
            close_chf = close_local.rename(name)
        else:
            fx = fx_map[f'{ccy}CHF']
            # Align FX to price dates and forward-fill FX only (never prices)
            before = fx.reindex(close_local.index)
            fx_aligned = before.ffill()
            filled = int(before.isna().sum() - fx_aligned.isna().sum())
            if DEBUG and filled > 0:
                print(f"    {name}: FX ffill filled {filled} missing FX points on price dates")
            close_chf = (close_local * fx_aligned).dropna().rename(name)

        chf_close[name] = close_chf
        print('---------------------------------------------')
    # Align on common dates and restrict to lookback window
    prices_df = pd.DataFrame(chf_close).dropna()
   
    if DEBUG and not prices_df.empty:
        print(
            f">>>Common window: {prices_df.index.min().date()} → {prices_df.index.max().date()} "
            f"({len(prices_df)} rows before tail)"
        )

    prices_df = prices_df.tail(lookback_days)
    rets_df = np.log(prices_df / prices_df.shift(1)).dropna()

    if prices_df.shape[0] < (lookback_days * 0.9):
        if DEBUG and series_ranges:
            # Identify limiting start date and which assets cause it
            try:
                limiting_start = max(s for s, _, _ in series_ranges.values() if s is not None)
                limiting_names = [name for name, (s, _, _) in series_ranges.items() if s == limiting_start]
                print(f"Limiting start date: {limiting_start.date()} by {', '.join(limiting_names)}")
            except Exception:
                pass
        print(
            f"After alignment only {prices_df.shape[0]} rows remain "
            f"(expected {lookback_days}). Data source may not have full history."
        )
    
    if rets_df.isna().any().any():
        raise ValueError("NaNs remained in returns after shift/drop; check data alignment.")
    if (prices_df <= 0).any().any():
        raise ValueError("Non-positive prices encountered; check source data.")

    values = {}
    for h in holdings:
        name = h['name']
        if h.get("type", "").lower() == "cash":
            if h.get('ccy', '').upper() != 'CHF':
                chfval = h['amount'] * fx_map.get(f"{h.get('ccy','').upper()}CHF", pd.Series([np.nan])).iloc[-1]
                values[name] = chfval
            else:
                values[name] = h['amount']
            continue
        last_price = chf_close[name].iloc[-1]
        values[name] = h['position'] * last_price

    total_val = sum(values.values())
    print(f"Total portfolio value (CHF): {total_val:.2f}")



    # Weights (CHF)
    weights = pd.Series()
    for h in holdings:
        name = h["name"]
        value = float(values[h["name"]])
        weight = value / total_val
        print(f"    {name}: value {value:.2f} CHF, weight {weight:.4%}")
        weights[name] = weight
    

    if not np.isclose(weights.sum(), 1.0, atol=1e-6):
        raise ValueError(f"Weights must sum to 1. Got {weights.sum():.6f}" "check postions input in holdings.")

    return rets_df, prices_df, weights



def portfolio_risk(rets_df: pd.DataFrame, weights: pd.Series) -> dict:
    """
    Compute annualized vols, correlation, covariance, portfolio vol,
    marginal risk contribution (MRC), and percent risk contribution (PRC).
    """
    # Annualized stats
    cov_daily = rets_df.cov()
    cov_annual = cov_daily * 252.0
    vol_ann = rets_df.std() * np.sqrt(252.0)
    corr = rets_df.corr()

    # Align weights
    w = weights.reindex(rets_df.columns).astype(float)
    Sigma_w = cov_annual @ w
    port_var = float(w @ Sigma_w)
    port_vol = float(np.sqrt(port_var)) if port_var > 0 else 0.0

    # Contributions
    mrc = Sigma_w / port_vol if port_vol > 0 else Sigma_w * 0.0
    prc = (w * Sigma_w) / port_var if port_var > 0 else w * 0.0

    summary = pd.DataFrame({
        "Weight": w,
        "Vol_1Y_CHF": vol_ann,
        "MRC": mrc,           # ∂σ_p/∂w_i (absolute marginal contribution)
        "PRC_%": prc * 100.0  # percent contribution to total variance (sums ~100%)
    }).sort_values("PRC_%", ascending=False)

    return {
        "port_vol": port_vol,
        "cov_annual": cov_annual,
        "corr": corr,
        "vol_ann": vol_ann,
        "mrc": mrc,
        "prc": prc,
        "summary": summary,
    }

## RISK MATRIX ##

In [ ]:
holdings0 = [
    {"name":"Unilever", "symbol":"ULVR.LON", "ccy":"GBP", "gbx":True,  "value_chf": 25000},
    {"name":"Shell",    "symbol":"SHEL.LON", "ccy":"GBP", "gbx":True,  "value_chf": 13000},
    {"name":"NatWest",  "symbol":"NWG.LON",  "ccy":"GBP", "gbx":True,  "value_chf":  5000},
    {"name":"Barclays", "symbol":"BARC.LON", "ccy":"GBP", "gbx":True,  "value_chf":  5000},
    {"name":"Tesco",    "symbol":"TSCO.LON", "ccy":"GBP", "gbx":True,  "value_chf":  5000},
    {"name":"SWDA",     "symbol":"SWDA.LON", "ccy":"GBP", "gbx":True,  "value_chf":  12000},
    {"name":"EMIM",     "symbol":"EMIM.LON", "ccy":"GBP", "gbx":True,  "value_chf":  8000},
    {"name":"IBM",      "symbol":"IBM",      "ccy":"USD", "gbx":False, "value_chf":  4000},
    {"name":"ERNS",     "symbol":"ERNS.LON", "ccy":"GBP", "gbx":True,  "value_chf":  5000},
]
IBKR = [
    {"name":"EMIM",     "symbol":"EMIM.LSE", "ccy":"GBP", "gbx":True, "position": 100},
    {"name":"ERNS",     "symbol":"ERNS.LSE", "ccy":"GBP", "gbx":False, "position": 119, 'risk_fx': "none"},
    # {"name":"CASH",     "symbol":"ERNS.LSE", "ccy":"GBP", "gbx":False, "position": 119, 'risk_fx': "none"},
    {"name":"IBM",     "symbol":"IBM.US", "ccy":"USD", "gbx":False, "position": 4},

    {"name":"SGLN",      "symbol":"SGLN.LSE",      "ccy":"GBP", "gbx":True, "position": 60},

    {"name":"VUAG",     "symbol":"VUAG.LSE", "ccy":"GBP", "gbx":False, "position": 28},
    {"name":"WSML",     "symbol":"WSML.LSE", "ccy":"USD", "gbx":False, "position": 313},
    # {"name": "NUCL", "symbol": "NUCL.SW", "ccy": "CHF", "gbx":False, "position": 59},
    # {"name": "URNM", "symbol": "URNM.LSE", "ccy": "USD", "gbx":False, "position": 210},
    {"name": "YCA", "symbol": "YCA.LSE", "ccy": "GBP", "gbx":True, "position": 350},
    # {"name":"XMWX",     "symbol":"XMWX.LSE", "ccy":"GBP", "gbx":False, "position": 125,},
    {"name":"VEU",     "symbol":"VEU.US", "ccy":"USD", "gbx":False, "position": 67,},
    # {"name":"INRG",     "symbol":"INRG.LSE", "ccy":"GBP", "gbx":True, "position": 314,},
    # {"name":"IH2O",     "symbol":"IH2O.LSE", "ccy":"GBP", "gbx":True, "position": 36,},
    # {"name":"REMX",     "symbol":"REMX.SW", "ccy":"CHF", "gbx":False, "position": 55,},
    {"name":"IFRA",     "symbol":"IFRA.US", "ccy":"USD", "gbx":False, "position": 20,},

    {"name": "CASH_CHF", "type": "cash", "ccy": "CHF", "amount": 13333},
    {"name": "CASH_GBP", "type": "cash", "ccy": "GBP", "amount": -12400},
    {"name": "CASH_USD", "type": "cash", "ccy": "USD", "amount": -2584},

]
holdings1 =[
    {"name": "CASH_CHF", "type": "cash", "ccy": "CHF", "amount": 15800},
    {"name": "CASH_GBP", "type": "cash", "ccy": "GBP", "amount": -12400},
    {"name":"XMWX",     "symbol":"XMWX.LSE", "ccy":"GBP", "gbx":False, "position": 125,},
    {"name":"ERNS",     "symbol":"ERNS.LSE", "ccy":"GBP", "gbx":False, "position": 119, 'risk_fx': "none"},
]
holdings = IBKR
MAX_AGE = 24
PERIOD = 1260
DEBUG = False
START = '2020-01-01'

rets_df, prices_df, w = build_returns_matrix_in_chf(holdings, lookback_days=PERIOD, max_age=MAX_AGE, no_fx=False, usd_shift=False, DEBUG=DEBUG)

# print(f'rets_df: {rets_df.tail()}')
# print(f'prices_df: {prices_df.tail()}')

risk = portfolio_risk(rets_df, w)

# print(rets_df.tail())
print("Portfolio σ (annualized, CHF): {:.2%}".format(risk["port_vol"]))
print(risk["summary"].round({"Weight":3,"Vol_1Y_CHF":3,"MRC":3,"PRC_%":1, }))
# Optional:
print(f'CORRELATION:')
print(f'{risk["corr"].round(2)}')
print(f'COVARIANCE:')
print(f'{risk["cov_annual"]}')

-------------fetching currencies-------------
GBPCHF.FOREX - using cached data
USDCHF.FOREX - using cached data
-------------fetching assets-------------
EMIM.LSE - using cached data
---------------------------------------------
ERNS.LSE - using cached data
    Skipping FX conversion for ERNS
---------------------------------------------
IBM.US - using cached data
---------------------------------------------
SGLN.LSE - using cached data
---------------------------------------------
VUAG.LSE - using cached data
---------------------------------------------
WSML.LSE - using cached data
---------------------------------------------
YCA.LSE - using cached data
---------------------------------------------
VEU.US - using cached data
---------------------------------------------
IFRA.US - using cached data
---------------------------------------------
Total portfolio value (CHF): 29231.85
    EMIM: value 3417.43 CHF, weight 11.6908%
    ERNS: value 12104.68 CHF, weight 41.4092%
    IBM: val

## SEARCH EOD ##

In [26]:
def eod_search(quey: str, token: str):
    import requests, pandas as pd
    url = f"https://eodhd.com/api/search/{quey}?api_token={token}&fmt=json"
    r = requests.get(url, timeout=30); r.raise_for_status()
    hits = r.json()
    # Return a small table to pick from
    return pd.DataFrame([{
        "code": h.get("Code"),
        "exchange": h.get("Exchange"),
        "name": h.get("Name"),
        "currency": h.get("Currency"),
        "type": h.get("Type"),
        "startdate": h.get("StartDate"),
        # earliet date

    } for h in hits])

# Usage:
df = eod_search("YCA", EOD_API)
# pick the line with the longest available history (often XETRA/LSE/SIX)
print(df)

           code exchange                                               name  \
0           YCA      LSE                                    Yellow Cake PLC   
1  LU1859217272   EUFUND  JSS Investmentfonds II Sicav- Sustainable Equi...   
2  LU0807706857   EUFUND     YCAP Fund - YCAP Tactical Investment A EUR Acc   
3  LU0807707749   EUFUND     YCAP Fund - YCAP Tactical Investment B EUR Acc   

  currency          type startdate  
0      GBX  Common Stock      None  
1      EUR          FUND      None  
2      EUR          FUND      None  
3      EUR          FUND      None  


## GET THE EARLIEST DATE ##

In [15]:
START = '2020-01-02'
ticker = f'URNM.LSE'
params = {
    'from': START, 
    'to': today,
    'api_token': EOD_API    
}
url_fx = f'https://eodhd.com/api/eod/{ticker}'

# Fetch EODHD daily FX and build a Series
fx_df = fetch_csv_robust(url_fx, params=params, ticker=ticker, max_age=24)

URNM.LSE - downloading fresh data
saving  cache/URNM.LSE.csv


## Equity correlation drift check
We’ll:
- compute a 60-day rolling average pairwise correlation across the equity ETFs in your `rets_df`.
- show the 1-year view you’re using now (limited by `XMWX`).
- also compute a 3-year view excluding `XMWX` (if data exist), to see whether the increase is structural or just recent.

In [23]:
# 1) Current 1-year view (already in rets_df)
# Pick equity ETFs present in rets_df
all_cols = list(rets_df.columns)
EQUITY_LIKE = [c for c in ["EMIM","VUAG","WSML","XMWX","IBM"] if c in all_cols]

if len(EQUITY_LIKE) >= 2:
    rets_eq_1y = rets_df[EQUITY_LIKE]
    # 60D rolling average of pairwise correlation (off-diagonal mean)
    def offdiag_mean(corr_m):
        if corr_m.shape[0] < 2:
            return np.nan
        n = corr_m.shape[0]
        return (corr_m.values.sum() - n) / (n*(n-1))

    roll_avg_corr_1y = (
        rets_eq_1y.rolling(60).corr().groupby(level=0).apply(lambda c: offdiag_mean(c))
    )

    print("1Y rolling(60) avg pairwise equity corr (tail):")
    print(roll_avg_corr_1y.dropna().tail(10))
else:
    print("Not enough equity-like tickers to compute 1Y pairwise correlation.")

# 2) Longer 3Y view excluding XMWX (if data available): re-fetch or extend window is out of scope here,
# but we can approximate by checking if older data exist in prices_df; if not, we demonstrate exclusion-only.
try:
    # If you want to explicitly exclude XMWX to avoid its shorter history limiting the window
    EQUITY_NO_XMWX = [c for c in EQUITY_LIKE if c != "XMWX"]
    if len(EQUITY_NO_XMWX) >= 2:
        # Use available rets_df (1Y). For a true 3Y view, rerun build_returns_matrix_in_chf with lookback_days=756.
        rets_eq_ex = rets_df[EQUITY_NO_XMWX]
        roll_avg_corr_ex = (
            rets_eq_ex.rolling(60).corr().groupby(level=0).apply(lambda c: offdiag_mean(c))
        )
        print("Ex-XMWX rolling(60) avg pairwise equity corr (tail):")
        print(roll_avg_corr_ex.dropna().tail(756))
    else:
        print("Not enough non-XMWX equity-like tickers to compute ex-XMWX correlation.")
except Exception as e:
    print("Correlation analysis note:", e)


1Y rolling(60) avg pairwise equity corr (tail):
Date
2025-09-02    0.549627
2025-09-03    0.551725
2025-09-04    0.549764
2025-09-05    0.545939
2025-09-08    0.545741
2025-09-09    0.543770
2025-09-10    0.529330
2025-09-11    0.523707
2025-09-12    0.510802
2025-09-15    0.515767
dtype: float64
Ex-XMWX rolling(60) avg pairwise equity corr (tail):
Date
2024-12-02    0.426043
2024-12-03    0.410413
2024-12-04    0.410285
2024-12-05    0.417191
2024-12-06    0.403619
                ...   
2025-09-09    0.506077
2025-09-10    0.487784
2025-09-11    0.483393
2025-09-12    0.470275
2025-09-15    0.477077
Length: 192, dtype: float64


## 3-year view excluding XMWX
We will rebuild returns with a longer lookback (756 trading days ≈ 3y) and exclude `XMWX` so the window isn’t truncated, then recompute the 60D rolling average pairwise correlation.